In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/real/traffic.csv")
df["DateTime"] = pd.to_datetime(df["DateTime"])

print(df.head())
print(df.shape)
print(df.groupby("Junction")["DateTime"].agg(["min", "max", "count"]))
print("Duplicate rows:", df.duplicated(["DateTime", "Junction"]).sum())
print(df.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/real/traffic.csv'

In [2]:
import os
print("Working folder:", os.getcwd())
path = "../data/raw/real"
print("Files in", path, ":", os.listdir(path) if os.path.exists(path) else "FOLDER NOT FOUND")

Working folder: C:\Users\anive\TRAFFIC-PROJECT\notebooks
Files in ../data/raw/real : []


In [1]:
import os
import pandas as pd

folder = "../data/raw/real"
files = [f for f in os.listdir(folder) if f.endswith((".csv", ".csv.gz"))]
print("Data files found:", files)

df = pd.read_csv(os.path.join(folder, files[0]))
df["date_time"] = pd.to_datetime(df["date_time"])

print(df.shape)
print(df[["date_time", "traffic_volume"]].head())
print("Date range:", df["date_time"].min(), "to", df["date_time"].max())
print("Duplicate timestamps:", df["date_time"].duplicated().sum())

Data files found: ['Metro_Interstate_Traffic_Volume.csv', 'traffic.csv']
(48204, 9)
            date_time  traffic_volume
0 2012-10-02 09:00:00            5545
1 2012-10-02 10:00:00            4516
2 2012-10-02 11:00:00            4767
3 2012-10-02 12:00:00            5026
4 2012-10-02 13:00:00            4918
Date range: 2012-10-02 09:00:00 to 2018-09-30 23:00:00
Duplicate timestamps: 7629


In [2]:
dup_times = df[df["date_time"].duplicated(keep=False)].sort_values("date_time")
print(dup_times[["date_time", "traffic_volume", "weather_main"]].head(8))

spread = df.groupby("date_time")["traffic_volume"].agg(["min", "max", "count"])
print("\nHours with more than one row:", (spread["count"] > 1).sum())
print("Of those, hours where traffic_volume differs:", (spread["min"] != spread["max"]).sum())

hourly = df.groupby("date_time")["traffic_volume"].mean()
full_range = pd.date_range(hourly.index.min(), hourly.index.max(), freq="h")
print("\nUnique hours present:", len(hourly))
print("Hours in the full range:", len(full_range))
print("Missing hours:", len(full_range) - len(hourly))

              date_time  traffic_volume weather_main
178 2012-10-10 07:00:00            6793         Rain
179 2012-10-10 07:00:00            6793      Drizzle
180 2012-10-10 08:00:00            6283         Rain
181 2012-10-10 08:00:00            6283      Drizzle
182 2012-10-10 09:00:00            5680         Rain
183 2012-10-10 09:00:00            5680      Drizzle
269 2012-10-14 09:00:00            2685      Drizzle
270 2012-10-14 09:00:00            2685         Mist

Hours with more than one row: 5445
Of those, hours where traffic_volume differs: 0

Unique hours present: 40575
Hours in the full range: 52551
Missing hours: 11976


In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. One row per hour (duplicates only differed in the weather label)
hourly = (df.drop_duplicates("date_time")
            .set_index("date_time")["traffic_volume"].sort_index())

# 2. Full hourly timeline; missing hours stay empty (not filled in)
full_index = pd.date_range(hourly.index.min(), hourly.index.max(), freq="h")
hourly = hourly.reindex(full_index)

# 3. Features: current hour, recent hours, and the same hour as the target yesterday
data = pd.DataFrame({"current": hourly})
for lag in [1, 2, 3, 23, 24]:
    data[f"lag_{lag}"] = hourly.shift(lag)
data["hour"] = data.index.hour
data["day_of_week"] = data.index.dayofweek
data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)
data["month"] = data.index.month
data["target"] = hourly.shift(-1)          # traffic in the NEXT hour
data = data.dropna()

# 4. Chronological split: earlier 80% to train, last 20% to test
split = int(len(data) * 0.8)
feature_cols = [c for c in data.columns if c != "target"]
X_train, X_test = data[feature_cols].iloc[:split], data[feature_cols].iloc[split:]
y_train, y_test = data["target"].iloc[:split], data["target"].iloc[split:]

rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

def report(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    print(f"{name}: MAE {mae:.1f} | RMSE {rmse:.1f} | R2 {r2:.3f}")

print("Usable rows:", len(data), "| train:", len(X_train), "| test:", len(X_test))
print("Test period:", y_test.index.min(), "to", y_test.index.max())
print("Average hourly volume in test set:", round(y_test.mean()))
print()
report("Baseline: current hour", y_test, X_test["current"])
report("Baseline: same hour yesterday", y_test, X_test["lag_23"])
report("Random Forest", y_test, rf.predict(X_test))

Usable rows: 33806 | train: 27044 | test: 6762
Test period: 2017-12-19 00:00:00 to 2018-09-30 22:00:00
Average hourly volume in test set: 3322

Baseline: current hour: MAE 582.9 | RMSE 805.5 | R2 0.831
Baseline: same hour yesterday: MAE 564.4 | RMSE 1019.5 | R2 0.729
Random Forest: MAE 148.6 | RMSE 231.6 | R2 0.986


In [4]:
train = data.iloc[:split]
test = data.iloc[split:]

# Typical value for each weekday + hour, learned from training data only
t_train = train.index + pd.Timedelta(hours=1)
profile = (pd.Series(train["target"].values, index=t_train)
           .groupby([t_train.dayofweek, t_train.hour]).mean())
t_test = test.index + pd.Timedelta(hours=1)
pred_profile = pd.Series([profile.get((d, h), np.nan)
                          for d, h in zip(t_test.dayofweek, t_test.hour)],
                         index=y_test.index)
report("Baseline: typical value for that weekday and hour", y_test, pred_profile)

# Same hour last week (where available)
last_week = hourly.shift(167).reindex(y_test.index)
ok = last_week.notna()
report(f"Baseline: same hour last week ({ok.sum()} rows)", y_test[ok], last_week[ok])
report("Random Forest on the same rows", y_test[ok], rf.predict(X_test)[ok.values])

Baseline: typical value for that weekday and hour: MAE 278.1 | RMSE 495.1 | R2 0.936
Baseline: same hour last week (6742 rows): MAE 350.7 | RMSE 666.2 | R2 0.884
Random Forest on the same rows: MAE 148.9 | RMSE 231.9 | R2 0.986


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

kg = pd.read_csv("../data/raw/real/traffic.csv")
kg["DateTime"] = pd.to_datetime(kg["DateTime"])
print(kg.groupby("Junction")["DateTime"].agg(["min", "max", "count"]))

def evaluate_junction(series, name):
    series = series.reindex(pd.date_range(series.index.min(), series.index.max(), freq="h"))
    d = pd.DataFrame({"current": series})
    for lag in [1, 2, 3, 23, 24]:
        d[f"lag_{lag}"] = series.shift(lag)
    d["hour"] = d.index.hour
    d["day_of_week"] = d.index.dayofweek
    d["is_weekend"] = (d["day_of_week"] >= 5).astype(int)
    d["target"] = series.shift(-1)          # next hour
    d = d.dropna()
    split = int(len(d) * 0.8)
    cols = [c for c in d.columns if c != "target"]
    tr, te = d.iloc[:split], d.iloc[split:]

    rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
    rf.fit(tr[cols], tr["target"])

    # typical value for weekday + hour, learned from training data only
    t_tr = tr.index + pd.Timedelta(hours=1)
    profile = (pd.Series(tr["target"].values, index=t_tr)
               .groupby([t_tr.dayofweek, t_tr.hour]).mean())
    t_te = te.index + pd.Timedelta(hours=1)
    typ = pd.Series([profile.get((a, b), np.nan) for a, b in zip(t_te.dayofweek, t_te.hour)],
                    index=te.index)
    ok = typ.notna()

    y = te["target"]
    return {"junction": name,
            "rows_train/test": f"{len(tr)}/{len(te)}",
            "avg_volume": round(y.mean(), 1),
            "MAE_current": round(mean_absolute_error(y, te["current"]), 2),
            "MAE_typical": round(mean_absolute_error(y[ok], typ[ok]), 2),
            "MAE_RandomForest": round(mean_absolute_error(y, rf.predict(te[cols])), 2),
            "R2_RandomForest": round(r2_score(y, rf.predict(te[cols])), 3)}

results = []
for j in [1, 2, 3]:
    s = (kg[kg["Junction"] == j].drop_duplicates("DateTime")
           .set_index("DateTime")["Vehicles"].sort_index())
    results.append(evaluate_junction(s, f"Junction {j}"))

pd.DataFrame(results)

                min                 max  count
Junction                                      
1        2015-11-01 2017-06-30 23:00:00  14592
2        2015-11-01 2017-06-30 23:00:00  14592
3        2015-11-01 2017-06-30 23:00:00  14592
4        2017-01-01 2017-06-30 23:00:00   4344


,junction,rows_train/test,avg_volume,MAE_current,MAE_typical,MAE_RandomForest,R2_RandomForest
0,Junction 1,11653/2914,67.2,6.26,27.51,4.48,0.926
1,Junction 2,11653/2914,23.7,2.94,11.76,3.15,0.742
2,Junction 3,11653/2914,17.5,3.72,5.98,3.08,0.698
